# 06 · Logistic regression baseline

Stage 1 model: interpretable, fast, minimum reference point. Trained on v1 train, selected on v1 validation, reported on v1 test. Holdout stays unused.

In [1]:
import pandas as pd

from cross_model_drift.data import load_split
from cross_model_drift.features import target_vector
from cross_model_drift.metrics import quality_metrics
from cross_model_drift.models import train_logistic
from cross_model_drift.notebook import setup_model_session
from cross_model_drift.tracking import clearml_task, log_metrics, log_parameters

nb = setup_model_session()
train = load_split("v1", "train", nb.config, engine=nb.engine)
valid = load_split("v1", "validation", nb.config, engine=nb.engine)
test = load_split("v1", "test", nb.config, engine=nb.engine)
len(train), len(valid), len(test)

(4981627, 546677, 534680)

In [2]:
y_train = target_vector(train)
model = train_logistic(train, y_train, threshold=nb.threshold)
valid_metrics = quality_metrics(target_vector(valid), model.predict_proba(valid), threshold=nb.threshold)
test_metrics = quality_metrics(target_vector(test), model.predict_proba(test), threshold=nb.threshold)
pd.DataFrame([valid_metrics, test_metrics], index=["validation", "test"])

,precision,recall,f1,pr_auc,roc_auc,threshold,n,n_positive,positive_rate
validation,0.041880,0.796577,0.079576,0.316484,0.819447,0.5,546677,11100,0.020304
test,0.043794,0.803487,0.083061,0.329192,0.820302,0.5,534680,11643,0.021776


In [3]:
path = model.save(nb.artifacts / "models" / "v1_logistic.joblib")
with clearml_task("train_logistic_baseline", config=nb.config, task_type="training", tags=["v1", "logistic"], init=True) as task:
    log_parameters(task, model.params)
    log_metrics(task, test_metrics, title="v1_test")
path

ClearML Task: created new task id=036fc67f2e364bf693c7a96441ba869c
ClearML results page: http://localhost:8080/projects/23d7eaf499d345e2b9df79587a66a66c/tasks/036fc67f2e364bf693c7a96441ba869c/output/log
2026-08-23 19:17:08,342 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring


PosixPath('/Users/mitter/aventures/code/cross-model-drift/artifacts/models/v1_logistic.joblib')

Set `init=True` after `~/.clearml/clearml.conf` points at the local server if you want this run logged.